<p><font size="6" color='grey'> <b>

Generative KI. Verstehen. Anwenden. Gestalten.
</b></font> </br></p>

<p><font size="5" color='grey'> <b>
MCP - Model Context Protocol mit LangChain Agent
</b></font> </br></p>

---

In [ ]:
#@title 🔧 Umgebung einrichten{ display-mode: "form" }
!uv pip install --system -q git+https://github.com/ralf-42/GenAI.git#subdirectory=04_modul
!uv pip install --system -q fastmcp langchain-mcp-adapters uvicorn

# LangSmith Env-Vars vor LangChain-Imports setzen

# ── Stdlib ────────────────────────────────────────────────────────────────────
import os
os.environ["LANGSMITH_TRACING"] = "false"
os.environ["LANGSMITH_PROJECT"] = "M10-MCP-LangChain-Agent"
os.environ["LANGSMITH_ENDPOINT"] = "https://eu.api.smith.langchain.com"

# ── Projekt-Utilities ─────────────────────────────────────────────────────────
from genai_lib.utilities import (
    check_environment,
    get_ipinfo,
    setup_api_keys,
    mprint,
    mermaid
)

setup_api_keys(['OPENAI_API_KEY'], create_globals=False)
print()
check_environment()
print()
get_ipinfo()

# 1 | Intro

---

<p><font color='black' size="5">
Was ist MCP (Model Context Protocol)?
</font></p>

MCP ist ein **standardisiertes Protokoll**, das LLMs mit externen Tools und Datenquellen verbindet — vergleichbar mit einem universellen Standard für Tool-Integration. Der Kern ist Standardisierung: ein Protokoll für alle Tools, einmal erstellt und überall nutzbar, einfach erweiterbar durch neue Server. MCP ist für echte Produktionsanwendungen ausgelegt, nicht nur für Prototypen.

**Architektur:**
```
LangChain Agent
        ↓
MCP Client (langchain-mcp-adapters)
        ↓ HTTP POST (JSON-RPC 2.0)
MCP Server (FastMCP auf localhost:8000)
        ↓
Tools (days_between, add_days, weekday_of)
```

<p><font color='black' size="5">
Wie funktioniert MCP?
</font></p>

MCP basiert auf einem **Client-Server-Modell** mit drei Rollen:

**MCP Server** stellt Tools (Funktionen) und Ressourcen (Daten) bereit, implementiert das MCP-Protokoll (JSON-RPC 2.0) und läuft als eigenständiger Prozess.

**MCP Client** verbindet sich mit einem oder mehreren Servern, lädt die verfügbaren Tools und übersetzt LLM-Anfragen in MCP-Protokoll.

**LLM / Agent** entscheidet, welche Tools benötigt werden, orchestriert mehrere Tool-Aufrufe und generiert die finale Antwort in natürlicher Sprache.

**Transport:**  — HTTP-basiert, für Web-Services und Notebook-Umgebungen. ✅ In diesem Notebook

✏️ Vergleich der MCP Transport-Optionen

<details>



| Transport | Beschreibung | Funktionsweise | Primärer Use-Case |
| --- | --- | --- | --- |
| **`stdio`** | **Standard Input/Output** | Daten werden über die Konsole (`stdin`/`stdout`) ausgetauscht. | **Lokale Integration:** Wenn der LLM-Client (z.B. Claude Desktop) den Server direkt als Prozess startet. |
| **`streamable_http`** | **HTTP-basiert** | Nutzt Standard `HTTP POST` Anfragen für den Austausch von JSON-RPC Objekten. | **Web-Services:** Ideal für Cloud-Infrastrukturen und Notebook-Umgebungen (wie in diesem Beispiel). |
| **`sse`** | **Server-Sent Events** | Ein unidirektionaler Stream vom Server zum Client über eine offene HTTP-Verbindung. | **Streaming:** Wenn der Server Antworten in Echtzeit "pushen" muss, ohne auf neue Anfragen zu warten. |
| **`websocket`** | **WebSockets** | Eine dauerhafte, voll-bidirektionale Verbindung zwischen beiden Parteien. | **Hochperformante Apps:** Wenn minimale Latenz und ständiger Datenaustausch in beide Richtungen nötig sind. |

</details>


✏️ MCP-Infrastruktur: Von der Verbindung zum Befehl

<details>

   

1. Das Fundament: **HTTP (Transport)**
Bevor Daten fließen, wird eine Verbindung via HTTP (Hypertext Transfer Protocol) hergestellt.

    + Der Server als Web-Service: Erreichbar unter einer festen Adresse (URL).

    + Die Methode: Nutzt HTTP POST, um Datenpakete gezielt an den Server zu senden.

    + Vorteil: Funktioniert überall (Firewall-freundlich) und ist der Standard für moderne Software-Kommunikation.

2. Der Inhalt: **JSON-RPC 2.0 (Protokoll)**
Innerhalb des HTTP-Transports wird die "Sprache" JSON-RPC gesprochen. Sie strukturiert den eigentlichen Befehl.

    + Anfrage: Enthält die id (Zuordnung), die method (Tool-Name) und params (Eingabewerte).

    + Antwort: Liefert das result oder eine Fehlermeldung zurück an den Agenten.

    + Vorteil: Schlank und präzise – es wird nur das übertragen, was für den Funktionsaufruf nötig ist.

3. Kurzübersicht: Die Hierarchie        
    + Transport-Layer (HTTP): Der LKW, der das Paket von A nach B bringt.
    + Payload-Layer (JSON-RPC): Der Inhalt des Pakets (der eigentliche Arbeitsauftrag).

Fazit: Erst wenn der HTTP-Kanal steht, kann JSON-RPC die Tools des MCP-Servers steuern.


</details>

✏️ Model Context Protocol (MCP) – JSON-RPC Kommunikation

<details>

Das MCP nutzt **JSON-RPC 2.0 über HTTP**. In der Praxis bedeutet das:

1. Der **Client** sendet eine `HTTP POST`-Anfrage an den Server.
2. Der **Body** enthält ein JSON-Objekt nach der JSON-RPC 2.0 Spezifikation.
3. Der **Server** antwortet mit `HTTP 200 OK` und dem entsprechenden Ergebnis im Body.

---

**Beispiel: Tool-Aufruf (`add_days`)**

Stellen wir uns vor, ein LangChain Agent möchte das Tool `add_days` mit dem Datum `2026-03-15` und `90` Tagen aufrufen.

*1. Client an Server (Request)*

Der Client sendet eine Anfrage an `http://127.0.0.1:8000/mcp`. Im MCP-Protokoll ist der JSON-RPC-Methodenname nicht `add_days`, sondern `tools/call`. Der Tool-Name steht in `params.name`:

```json
{
  "jsonrpc": "2.0",
  "id": "some-unique-request-id",
  "method": "tools/call",
  "params": {
    "name": "add_days",
    "arguments": {
      "date": "2026-03-15",
      "days": 90
    }
  }
}
```

Erklärung der Felder:

* **`jsonrpc`**: Die Protokoll-Version (immer `"2.0"`).
* **`id`**: Eine eindeutige ID, um die Antwort später der richtigen Anfrage zuzuordnen.
* **`method`**: Die MCP-Operation. Für Tool-Aufrufe ist das `"tools/call"`.
* **`params.name`**: Der Name des aufzurufenden Tools (hier: `add_days`).
* **`params.arguments`**: Ein Objekt mit den Argumenten als Schlüssel-Wert-Paare (entspricht den Python-Parametern `date` und `days`).

---

*2. Server an Client (Response)*

Nach erfolgreicher Ausführung sendet der Server das Ergebnis zurück. MCP-Tool-Ergebnisse liegen im `result`-Objekt, typischerweise als `content` und optional als strukturierte Daten:

```json
{
  "jsonrpc": "2.0",
  "id": "some-unique-request-id",
  "result": {
    "content": [
      {
        "type": "text",
        "text": "2026-06-13"
      }
    ],
    "structuredContent": {
      "result": "2026-06-13"
    },
    "isError": false
  }
}
```

* **`result.content`**: Das unstrukturierte Tool-Ergebnis, das ein Agent direkt lesen kann.
* **`structuredContent`**: Optionales maschinenlesbares Ergebnis.
* **`id`**: Muss exakt mit der ID der Anfrage übereinstimmen.

---

Fehlerbehandlung:

Wenn die Anfrage selbst ungültig ist, kann ein JSON-RPC-`error` zurückkommen. Fehler innerhalb eines Tools werden in MCP aber üblicherweise als Tool-Ergebnis mit `isError: true` zurückgegeben:

```json
{
  "jsonrpc": "2.0",
  "id": "some-unique-request-id",
  "result": {
    "content": [
      {
        "type": "text",
        "text": "ValueError: Ungültiges Datumsformat: '15.03.2026' — erwartet wird YYYY-MM-DD"
      }
    ],
    "isError": true
  }
}
```

---

> Diese Datenstruktur wird über die `streamable_http`-Verbindung ausgetauscht. Tools wie der `langchain-mcp-adapters` Client und der `FastMCP` Server übernehmen die Serialisierung automatisch.

</details>

<p><font color='black' size="5">
Praktisches Beispiel: Was passiert?
</font></p>

Ein AI-Assistent hat die Aufgabe: **Welcher Wochentag ist 90 Tage nach dem 15.03.2026?**

**Ohne MCP:**
```
AI: "Das ist ein Montag."
```
→ KI rechnet im Kopf — bei Datumsarithmetik häufig falsch (Schaltjahre, Monatslängen, Wochentagszyklen)

**Mit MCP:**
```
1. AI: "Ich brauche das add_days-Tool"
2. MCP Client → Server: add_days("2026-03-15", 90)
3. Server → Client: "2026-06-13"
4. AI: "Ich brauche das weekday_of-Tool"
5. MCP Client → Server: weekday_of("2026-06-13")
6. Server → Client: "Samstag"
7. AI: "Der 13.06.2026 ist ein Samstag."
```
→ Echte Berechnungen über Tools: präzise, erweiterbar, wiederverwendbar und nachvollziehbar — jeder Tool-Call ist im Trace dokumentiert.

**Ressourcen:**
- [FastMCP Documentation](https://www.prefect.io/fastmcp)
- [LangChain MCP Adapters](https://github.com/langchain-ai/langchain-mcp-adapters)
- [MCP Official Docs](https://modelcontextprotocol.io/)


# 2 | FastMCP HTTP-Server starten

---

**FastMCP** ist ein Python-Framework, das die Erstellung von MCP-Servern (Model Context Protocol) radikal vereinfacht. Der Name ist angelehnt an FastAPI – die gleiche Philosophie: minimaler Boilerplate-Code (repetitiver, standardisierter Code, der für die Grundstruktur eines Programms nötig ist, aber keine eigentliche Geschäftslogik enthält), maximale Funktionalität.

**Kernidee**

```python
from datetime import date
from fastmcp import FastMCP

mcp = FastMCP("MeinServer")

@mcp.tool()
def days_between(start: str, end: str) -> int:
    """Anzahl der Tage zwischen zwei Daten (YYYY-MM-DD)."""
    return (date.fromisoformat(end) - date.fromisoformat(start)).days
```

Mit nur wenigen Zeilen wird eine Python-Funktion zu einem Tool, das jedes LLM über das MCP-Protokoll aufrufen kann.

**Hauptmerkmale**

| Feature | Beschreibung |
|---------|--------------|
| **Decorator-basiert** | `@mcp.tool()` macht jede Funktion zum MCP-Tool |
| **Automatische Schemas** | Type Hints werden zu JSON-Schemas für das LLM |
| **Mehrere Transports** | HTTP, stdio, SSE, WebSocket |
| **Docstrings → Beschreibungen** | Das LLM "liest" die Docstrings, um Tools zu verstehen |

**Warum FastMCP?**

Ohne FastMCP müsste man das JSON-RPC 2.0 Protokoll, Session-Management und Tool-Schemas manuell implementieren. FastMCP abstrahiert das alles – man schreibt nur die eigentliche Logik.

**Kurz:** FastMCP ist für MCP das, was FastAPI für REST-APIs ist – der schnellste Weg vom Python-Code zum produktionsreifen Server.

<p><font color='black' size="5">
Server erstellen
</font></p>

Es wird ein FastMCP-Server *erstellt*, der dann als **subprocess** gestartet wird.

In [ ]:
# ── Stdlib ────────────────────────────────────────────────────────────────────
import subprocess
import sys
import time

# ── Web & APIs ────────────────────────────────────────────────────────────────
import requests

# ── LangChain ─────────────────────────────────────────────────────────────────
from langchain.agents import create_agent
from langchain.chat_models import init_chat_model
from langchain_core.messages import HumanMessage
from langchain_mcp_adapters.client import MultiServerMCPClient

In [ ]:
# 🚀 MCP HTTP-Server: Konfiguration

SERVER_HOST = "127.0.0.1"
SERVER_PORT = 8000
MCP_PATH = "/mcp"
SERVER_BASE_URL = f"http://{SERVER_HOST}:{SERVER_PORT}"
MCP_URL = f"{SERVER_BASE_URL}{MCP_PATH}"

mprint("### 🚀 MCP HTTP-Server vorbereiten")
mprint("---")

In [ ]:
%%writefile date_mcp_server.py
"""FastMCP Calendar Server - HTTP Transport"""

# ── Stdlib ────────────────────────────────────────────────────────────────────
import os
from datetime import date, timedelta

# ── Web & APIs ────────────────────────────────────────────────────────────────
from fastmcp import FastMCP

# FastMCP Server initialisieren
mcp = FastMCP("CalendarTools")

WOCHENTAGE = [
    "Montag", "Dienstag", "Mittwoch", "Donnerstag",
    "Freitag", "Samstag", "Sonntag",
]

def _parse(datum: str) -> date:
    """Wandelt einen ISO-String (YYYY-MM-DD) in ein date-Objekt um."""
    try:
        return date.fromisoformat(datum)
    except ValueError:
        raise ValueError(f"Ungültiges Datumsformat: {datum!r} — erwartet wird YYYY-MM-DD")

@mcp.tool()
def days_between(start: str, end: str) -> int:
    """Anzahl der Tage zwischen zwei Daten (Format YYYY-MM-DD)."""
    return (_parse(end) - _parse(start)).days

@mcp.tool()
def add_days(date: str, days: int) -> str:
    """Addiert Tage zu einem Datum (Format YYYY-MM-DD) und gibt das neue Datum zurück."""
    return (_parse(date) + timedelta(days=days)).isoformat()

@mcp.tool()
def weekday_of(date: str) -> str:
    """Gibt den deutschen Wochentag eines Datums (Format YYYY-MM-DD) zurück."""
    return WOCHENTAGE[_parse(date).weekday()]

if __name__ == "__main__":
    # FastMCP nennt diesen Servertransport "http".
    # Der LangChain-Adapter verbindet sich damit als "streamable_http".
    # Host/Port/Pfad übergibt die Start-Zelle als Umgebungsvariablen —
    # einzige Quelle der Wahrheit ist die Konfigurations-Zelle des Notebooks.
    mcp.run(
        transport="http",
        host=os.environ.get("MCP_HOST", "127.0.0.1"),
        port=int(os.environ.get("MCP_PORT", "8000")),
        path=os.environ.get("MCP_PATH", "/mcp"),
    )

In [ ]:
# ▶️ MCP HTTP-Server starten

print("⏳ Starte Server als subprocess...")

# Prüfe, ob Port 8000 bereits belegt ist.
# Wenn ja, starten wir NICHT blind einen zweiten Prozess, weil sonst unklar wäre,
# welcher Server die späteren MCP-Anfragen beantwortet.
try:
    requests.get(SERVER_BASE_URL, timeout=1)
    print(f"⚠️ Port {SERVER_PORT} ist bereits belegt.")
    print("   Bitte führe zuerst die Cleanup-Zelle aus oder starte den Kernel neu.")
    raise RuntimeError(f"Port {SERVER_PORT} ist bereits belegt")
except requests.exceptions.RequestException:
    pass  # Kein Dienst erreichbar: Port ist für dieses Notebook frei.

# Host/Port/Pfad aus der Konfigurations-Zelle als Umgebungsvariablen übergeben.
# stdout/stderr werden gesammelt, damit Serverfehler im Notebook sichtbar bleiben.
server_process = subprocess.Popen(
    [sys.executable, "date_mcp_server.py"],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE,
    text=True,
    env={
        **os.environ,
        "MCP_HOST": SERVER_HOST,
        "MCP_PORT": str(SERVER_PORT),
        "MCP_PATH": MCP_PATH,
    },
)

print("⏳ Warte auf Server-Start...")
max_retries = 20
for _ in range(max_retries):
    if server_process.poll() is not None:
        stdout, stderr = server_process.communicate()
        if stdout:
            print(stdout)
        if stderr:
            print(stderr)
        raise RuntimeError("MCP-Server-Prozess wurde sofort beendet. Prüfe date_mcp_server.py.")

    try:
        # Nur prüfen, ob der HTTP-Server schon antwortet.
        # Die eigentliche MCP-Protokollprüfung folgt mit client.get_tools().
        requests.get(SERVER_BASE_URL, timeout=1)
        break
    except requests.exceptions.RequestException:
        pass  # Server noch nicht bereit — nach kurzer Pause erneut versuchen.
    time.sleep(1)
else:
    # Alle Versuche erschöpft: Prozess aufräumen und Fehler melden.
    print("❌ Server konnte nicht gestartet werden!")
    if server_process.poll() is None:
        server_process.terminate()
        try:
            stdout, stderr = server_process.communicate(timeout=5)
        except subprocess.TimeoutExpired:
            server_process.kill()
            stdout, stderr = server_process.communicate()
        if stdout:
            print(stdout)
        if stderr:
            print(stderr)
    raise RuntimeError("Server-Start fehlgeschlagen")

mprint(f"\n✅ MCP-Server läuft auf {MCP_URL}")
print(f"💡 Server-Prozess PID: {server_process.pid}")
print("💡 Der Server läuft als echter subprocess!")

<details>

<summary><strong>
Ports & localhost
</strong></summary

</details>


Beim Aufruf einer URL wie `http://127.0.0.1:8000/1/mcp` läuft intern folgender Prozess ab:

Der Rechner kommuniziert mit sich selbst (über `127.0.0.1`) und leitet die Anfrage an ein **konkretes Programm** weiter – identifiziert über den **Port**.

Da mehrere Programme gleichzeitig Netzwerkverbindungen nutzen (z. B. Browser, Datenbank, API), benötigt jedes Programm einen eigenen „Eingang“.
Dieser Eingang wird als **Port** bezeichnet.

---

**Kurzdefinitionen**

**`127.0.0.1` (localhost)**    
→ Eigener Rechner (Loopback)   
→ Kommunikation bleibt vollständig lokal   

**Ports (0–65535)**    
→ Virtuelle Eingänge für Programme    
→ Ermöglichen parallele Netzwerkkommunikation    

---

**Komplett-Beispiel (MCP)**

```id="k2y7vd"
http://127.0.0.1:8000/1/mcp
        └──────┬──────┘
               │
     Adresse + Port = Zielprogramm
```

| Komponente  | Bedeutung                                 | Beispiel     |
| ----------- | ----------------------------------------- | ------------ |
| `127.0.0.1` | Lokaler Rechner (nicht extern erreichbar) | Dev-Server   |
| `:8000`     | Port → welches Programm hört dort?        | FastAPI/MCP  |
| `/1/mcp`    | Route innerhalb der Anwendung             | API-Endpunkt |

---

**Wichtige Präzisierung**

* Ein **Port ist kein Programm**
* Ein Programm **bindet sich an einen Port**
* Beispiel:    
  → FastAPI auf Port 8000   
  → anderes Tool z. B. auf Port 3000    

---

**Mentales Modell**

* IP = **Hausadresse**
* Port = **Wohnung**
* Pfad = **Person in der Wohnung**

---

**Merksatz**

> `127.0.0.1` = Kommunikation mit dem eigenen Rechner     
> `:8000` = Weiterleitung an die Anwendung, die dort lauscht   

---

**Kritischer Zusatz**

„Lokal“ bedeutet lediglich: **nicht von außen erreichbar**      
→ keine Garantie für Sicherheit (z. B. bei unsicheren APIs)


# 3 | LangChain MCP Client

---

Der **MCPClient** verbindet sich über **streamable_http** mit dem Server:

In [ ]:
#@markdown Kommunikationsdiagramm

diagram = '''
%%{init: {'theme':'forest'}}%%
sequenceDiagram

    autonumber
    actor User
    participant Agent as LangChain Agent
    participant Client as MCP Client
    participant Server as MCP Server (FastMCP)

    Note over Agent,Server: Initialisierung (Handshake)

    Client->>Server: HTTP POST (method: initialize)
    Server-->>Client: JSON Response (Capabilities & Version)
    Client->>Server: HTTP POST (notifications/initialized)

    Note over Agent,Server: Tools abfragen (await client.get_tools())

    Client->>Server: HTTP POST (method: tools/list)
    Note right of Client: JSON-RPC 2.0 Request

    Server-->>Client: JSON Response (Liste der Tool-Definitionen)
    Note left of Server: Name, Description, Input Schema
'''
mermaid(diagram, width=800, height=550)

In [ ]:
#🔌 MCP Client konfigurieren und Tools laden

mprint("### 🔌 MCP Client Setup und Tool-Discovery")
mprint("---")

mprint("💡 Hinweis: FastMCP startet den Server mit `transport=\"http\"`.")
mprint("   Der LangChain-Adapter verbindet sich damit über `streamable_http`.")

mcp_config = {
    "calendar": {
        "transport": "streamable_http",
        "url": MCP_URL,
    }
}

try:
    client = MultiServerMCPClient(mcp_config)
    tools = await client.get_tools()

    if not tools:
        raise RuntimeError("Server erreichbar, aber keine Tools gefunden.")

    print("✅ MCP-Server ist online und spricht das MCP-Protokoll.")
    print(f"🌐 URL: {MCP_URL}")
    print(f"🔧 {len(tools)} Tools geladen:")
    for tool in tools:
        print(f"   - {tool.name}: {tool.description}")

except Exception as e:
    print(f"❌ MCP Client Setup fehlgeschlagen: {e}")
    print("   Prüfe, ob der Server läuft und ob MCP_URL korrekt gesetzt ist.")
    raise


# 4 | LangChain Agent

---

<p><font color='black' size="5">
Agent mit MCP-Tools erstellen
</font></p>

Der Agent kommuniziert über **streamable_http** mit dem Server!

In [ ]:
# 🤖 LangChain Agent erstellen

# ── Projekt-Utilities ─────────────────────────────────────────────────────────
from genai_lib.model_config import WORKER
mprint("### 🤖 LangChain Agent erstellen")
mprint("---")

# SCHRITT 1: LLM initialisieren (LangChain 1.0+ API)
print("1️⃣ LLM initialisieren...")
llm = init_chat_model(WORKER)
print("   ✅ Modell geladen\n")

# SCHRITT 2: Agent erstellen mit MCP-Tools
print("2️⃣ Agent erstellen...")

tool_list = "\n".join(
    f"- {tool.name}: {tool.description}"
    for tool in tools
)

system_prompt_agent = f"""

<Task>
Du bist ein hilfreicher Kalender-Assistent.
</Task>

<Instructions>
Du hast Zugriff auf folgende Datums-Tools über einen MCP HTTP-Server:
{tool_list}

Nutze diese Tools für alle Datumsberechnungen.
Übergib Datumsangaben an die Tools immer im Format YYYY-MM-DD.
Antworte immer auf Deutsch und erkläre deine Rechenschritte.
</Instructions>

<Hard Limits>
Berechne Datumsdifferenzen und Wochentage niemals im Kopf,
sondern ausschließlich mit den bereitgestellten Tools.
</Hard Limits>"""

agent = create_agent(
    model=llm,
    tools=tools,  # Tools vom MCP-Server (über streamable_http)
    system_prompt=system_prompt_agent,
    debug=False,  # Weniger Output
)
print("   ✅ Agent bereit\n")

mprint("🎯 Agent ist bereit für Queries!")
mprint("💡 Der Agent kommuniziert über streamable_http mit dem MCP-Server")

In [ ]:
async def run_agent_query(query: str):
    """Führt eine Agent-Abfrage aus und gibt Response und finale Nachricht zurück."""
    mprint(f"### 🤖 Query: {query}")
    mprint("---")

    response = await agent.ainvoke({
        "messages": [HumanMessage(content=query)]
    })
    final_message = response["messages"][-1]
    mprint(f"\n🤖 **Antwort:** {final_message.content}")
    return response, final_message

# 5 | Praktische Beispiele

---

In [ ]:
#@markdown Kommunikationsdiagramm

diagram = '''
%%{init: {'theme':'forest'}}%%
sequenceDiagram
    autonumber
    actor User
    participant Agent as LangChain Agent
    participant Client as MCP Client
    participant Server as MCP Server (FastMCP)

    User->>Agent: "Welcher Wochentag ist 90 Tage nach dem 15.03.2026?"
    Agent->>Client: Wählt Tool 'add_days("2026-03-15", 90)'

    Note over Client,Server: Transport: streamable_http

    Client->>Server: HTTP POST (JSON-RPC)
    activate Server
    Server->>Server: Führt Tool 'add_days' aus
    Server-->>Client: Return "2026-06-13"
    deactivate Server

    Client-->>Agent: Ergebnis: "2026-06-13"

    Note right of Agent: Agent entscheidet:<br/>Wochentag fehlt noch

    Agent->>Client: Wählt Tool 'weekday_of("2026-06-13")'
    Client->>Server: HTTP POST (JSON-RPC)
    activate Server
    Server->>Server: Führt Tool 'weekday_of' aus
    Server-->>Client: Return "Samstag"
    deactivate Server

    Client-->>Agent: Ergebnis: "Samstag"
    Agent->>User: "Der 13.06.2026 ist ein Samstag"
'''
mermaid(diagram, width=800, height=550)

<p><font color='black' size="5">
Motivation: Datumsarithmetik ohne Tools
</font></p>

Datumsberechnungen sind eine bekannte LLM-Schwäche: Schaltjahre, unterschiedliche Monatslängen und Wochentagszyklen laden zum Verrechnen ein. Die folgende Zelle stellt dem LLM dieselbe Frage wie später Demo #3 — aber **ohne Tools**.

Die Antwort kann richtig sein, ist aber weder garantiert noch nachvollziehbar. Mit MCP-Tools wird die Berechnung deterministisch, und jeder Schritt ist im Trace sichtbar.

<p><font color='black' size="5">
Einfache Datumsabfragen
</font></p>

In [ ]:
#🤖 1: Datumsdifferenz (Neujahr → Heiligabend)

query = "Wie viele Tage liegen zwischen dem 01.01.2026 und dem 24.12.2026?"

response, final_message = await run_agent_query(query)

✏️ Details: Model Context Protocol (MCP) – JSON-RPC Kommunikation

<details>

Das JSON-RPC 2.0 Kommunikationsprotokoll ist in Kapitel 1 (Intro) detailliert beschrieben. Das Schema ist identisch — `method`, `params`, `id` im Request; `result` oder `error` in der Response.

> Der `langchain-mcp-adapters` Client und der `FastMCP` Server übernehmen die Serialisierung automatisch. Jeder Tool-Call in diesem Kapitel folgt demselben Muster.

</details>

In [ ]:
# 🤖 2: Wochentag (Tag der Deutschen Einheit 2026)

query = "Auf welchen Wochentag fällt der 3. Oktober 2026?"

response, final_message = await run_agent_query(query)

<p><font color='black' size="5">
Komplexe Datumsfragen (Multi-Tool)
</font></p>

Der Agent orchestriert mehrere HTTP-Calls zum Server:

In [ ]:
# 🤖 3: Verkettung — Datum berechnen, dann Wochentag

query = "Welcher Wochentag ist 90 Tage nach dem 15.03.2026? Nutze die MCP-Kalender-Tools."

response, final_message = await run_agent_query(query)

In [ ]:
# 🤖 4: Projektplanung — Enddatum und Wochentag

query = "Ein Projekt startet am 01.09.2026 und dauert 45 Tage. An welchem Datum endet es und auf welchen Wochentag fällt das Ende?"

response, final_message = await run_agent_query(query)

In [ ]:
#@markdown 🛑 MCP-Server beenden

mprint("### 🛑 MCP-Server Cleanup")
mprint("---")

if 'server_process' not in globals():
    print("⚠️ Keine server_process Variable gefunden")
elif server_process.poll() is not None:
    print(f"✅ Server-Prozess (PID {server_process.pid}) war bereits beendet")
else:
    server_process.terminate()
    try:
        server_process.wait(timeout=5)
        print(f"✅ Server-Prozess (PID {server_process.pid}) wurde beendet")
    except subprocess.TimeoutExpired:
        server_process.kill()
        server_process.wait()
        print(f"✅ Server-Prozess (PID {server_process.pid}) wurde hart beendet (kill)")

# A | Aufgaben
---

<p><font color='darkblue' size="4">
📌 <b>Wichtig</b>
</font></p>

Die Aufgabestellungen unten bieten Anregungen — eigene Herausforderungen sind ausdrücklich willkommen.

**Hinweis zur Lösungshilfe:**
> Generative KI darf und soll im Kurs auch als Lernunterstützung genutzt werden — z. B. Gemini in Google Colab, um Fehlermeldungen zu verstehen, Teilschritte zu klären oder Code-Varianten zu prüfen.
> Der Schwerpunkt des Kurses bleibt: GenAI-Apps selbst verstehen, aufbauen und gezielt weiterentwickeln.

<p><font color='black' size="5">
Eigene MCP-Tools hinzufügen
</font></p>

**Ziel:** Erweitere den MCP-Server mit neuen Tools.

**Aufgabe 1: Wochenend-Check**
Füge ein `is_weekend(date)` Tool hinzu, das `True`/`False` zurückgibt (bearbeite `date_mcp_server.py`, dann Server neu starten).

**Aufgabe 2: Werktage**
Füge ein `add_business_days(date, days)` Tool hinzu, das nur Montag bis Freitag zählt.

**Aufgabe 3: Komplexe Query**
Teste mit: `"Ein Projekt startet am 01.09.2026 und dauert 20 Werktage. Wann endet es, und fällt das Ende auf ein Wochenende?"`

**Schritte:**
1. Bearbeite die Server-Datei `date_mcp_server.py`
2. Starte Server neu (führe Cleanup-Zelle aus, dann Zelle 2 neu)
3. Teste mit eigenen Queries


---

**Grundlagen**

Rufe ein bestehendes MCP-Tool auf und dokumentiere das Ergebnis:

1. Zeige an, welche Tools der MCP-Server bereitstellt.
2. Rufe eines der verfügbaren Tools direkt auf (z. B. Wochentag oder Datumsdifferenz).
3. Gib das Ergebnis aus und kommentiere, was der Agent intern getan hat.

**✅ Erledigt wenn:** Das MCP-Tool wird aufgerufen und gibt ein Ergebnis zurück — sichtbar in der Ausgabe, nicht nur im Log.

In [ ]:
# Grundlagen: Bestehendes MCP-Tool aufrufen
# Startpunkt: MCP-Client aus Kapitel 3 verwenden

# 1. MCP-Server starten (Kapitel 2)
# 2. Agent mit MCP-Tool verbinden
# 3. Einfache Anfrage stellen
# 4. Ergebnis + internen Workflow ausgeben
# ...

**Aufbau**

Kombiniere mehrere MCP-Tools, um eine zusammengesetzte Aufgabe zu lösen:

1. Formuliere eine Anfrage, die mindestens zwei MCP-Tools erfordert (z. B. `add_days` + `weekday_of`).
2. Führe die Anfrage über den Agenten aus.
3. Protokolliere die Zwischenschritte (Tool-Calls) und das Endergebnis.

**✅ Erledigt wenn:** Die Ausgabe zeigt für jeden der mindestens zwei Tool-Aufrufe Eingabe, Zwischenschritt und Ergebnis.

In [ ]:
# Aufbau: Mehrere MCP-Tools kombinieren
# Startpunkt: MCP-Agent aus Kapitel 4 als Vorlage

# Zwei Tools die zusammenspielen (z. B. add_days + weekday_of)
# Zwischenschritte mit verbose=True oder Callbacks sichtbar machen
# ...

**Vertiefung**

Ergänze den MCP-Server um ein eigenes neues Tool und teste es:

1. Definiere eine neue Python-Funktion und registriere sie als MCP-Tool (z. B. Potenz, Modulo oder eine Textfunktion).
2. Starte den Server neu und lade die erweiterte Tool-Liste.
3. Führe eine Anfrage durch, die das neue Tool nutzt, und dokumentiere das Ergebnis.
4. Optional: Aktiviere LangSmith-Tracing und halte die Trace-URL fest.

**✅ Erledigt wenn:** Das neue Tool läuft im MCP-Server und gibt bei Aufruf ein korrektes Ergebnis zurück — manuell geprüft.

In [ ]:
# Vertiefung: MCP-Server mit neuem Tool erweitern
# Startpunkt: MCP-Server-Implementierung aus Kapitel 2

# 1. Neues Tool implementieren (z. B. is_weekend, add_business_days)
# 2. Im MCP-Server registrieren
# 3. Agent aufrufen und neues Tool testen
# ...

# B | Dokumente zum Weiterlesen
---




📚 Ergänzende Artikel aus der Kurs-Dokumentation:

- [Model Context Protocol](https://ralf-42.github.io/GenAI/08-agenten/mcp-model-context-protocol.html)
- [Agenten-Architekturen](https://ralf-42.github.io/GenAI/08-agenten/agent-architekturen.html)
- [Tool Use & Function Calling](https://ralf-42.github.io/GenAI/08-agenten/tool-use-function-calling.html)